In [ ]:
!pip install --upgrade transformers trl


In [ ]:
!pip install latex2sympy2_extended


In [2]:
# 基础库
import logging
import os
import sys
import re
import math
from dataclasses import dataclass, field
from typing import List, Optional

# PyTorch 和 Hugging Face Transformers
import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    TrainingArguments,
    set_seed,
    TrainerCallback,
    TrainerControl,
    TrainerState,
)
from transformers.trainer_utils import get_last_checkpoint


# 数据集工具
import datasets
from datasets import load_dataset

# TRL（Transformers Reinforcement Learning）库
from trl import (
    AutoModelForCausalLMWithValueHead, 
    PPOConfig, 
    PPOTrainer, 
    GRPOTrainer, 
    GRPOConfig, 
    SFTTrainer
)
# 数学公式处理与验证相关工具
from latex2sympy2_extended import NormalizationConfig
from math_verify import LatexExtractionConfig, parse, verify

/opt/anaconda3/envs/gym3.10.10/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/anaconda3/envs/gym3.10.10/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an 

trl 与 transformers的版本不兼容导致的：

RuntimeError: Failed to import trl.models.modeling_value_head because of the following error (look up to see its traceback):
cannot import name 'HybridCache' from 'transformers' (/opt/anaconda3/envs/gym3.10.10/lib/python3.10/site-packages/transformers/__init__.py)

In [3]:
# 加载数据集
dataset = load_dataset("../../datasets/NuminaMath-TIR","default")

# 查看数据集结构
print("数据集包含的字段",dataset)
# 获取训练集中的第一个样本
sample = dataset['train'][0]
print(json.dumps(sample,indent=2,ensure_ascii=False))

数据集包含的字段 DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 72441
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 99
    })
})
{
  "problem": "What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.",
  "solution": "To determine the coefficient of \\(x^2y^6\\) in the expansion of \\(\\left(\\frac{3}{5}x - \\frac{y}{2}\\right)^8\\), we can use the binomial theorem.\n\nThe binomial theorem states:\n\\[\n(a + b)^n = \\sum_{k=0}^{n} \\binom{n}{k} a^{n-k} b^k\n\\]\n\nIn this case, \\(a = \\frac{3}{5}x\\), \\(b = -\\frac{y}{2}\\), and \\(n = 8\\).\n\nWe are interested in the term that contains \\(x^2y^6\\). In the general term of the binomial expansion:\n\\[\n\\binom{8}{k} \\left(\\frac{3}{5}x\\right)^{8-k} \\left(-\\frac{y}{2}\\right)^k\n\\]\n\nTo get \\(x^2\\), we need \\(8 - k = 2\\), thus \\(

In [5]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 模型名称与输出目录
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "data/Qwen-GRPO-training"
# LOCAL_MODEL_PATH = "/home/gittao/models/Qwen2.5-0.5B-Instruct/"
# 创建输出目录（如不存在则新建）
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 初始化 tokenizer，使用聊天模板
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)


# 如果没有设置 pad_token，则使用 eos_token 代替
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 输出 tokenizer 相关信息
print(f"词表大小（Vocabulary size）: {len(tokenizer)}")
print(f"模型最大序列长度（Model max length）: {tokenizer.model_max_length}")
print(f"填充标记（Pad token）: {tokenizer.pad_token}")
print(f"结束标记（EOS token）: {tokenizer.eos_token}")

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16  # 使用 bfloat16 更节省显存
)

# 输出模型参数总量
print(f"模型参数总量: {model.num_parameters():,}")

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x151896170>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: e8897c45-c18e-4832-b343-d74c837d487b)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x151896fe0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: a1002c4a-5c1f-4870-8af3-e0489d7368f1)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenize

词表大小（Vocabulary size）: 151665
模型最大序列长度（Model max length）: 131072
填充标记（Pad token）: <|endoftext|>
结束标记（EOS token）: <|im_end|>


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json (Caused by SSLError(SSLZeroReturnError(6, 'TLS/SSL connection has been closed (EOF) (_ssl.c:997)')))"), '(Request ID: 87f65f38-246e-4863-ab52-c14df4a9c8d0)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json (Caused by SSLError(SSLZeroReturnError(6, 'TLS/SSL connection has been closed (EOF) (_ssl.c:997)')))"), '(Request ID: 26760e54-681d-4253-8bd8-b4b900baa440)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/

模型参数总量: 494,032,768


In [ ]:
# 检查设备并加载模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备：{device}")
model.to(device)

# 定义基础推理测试函数
def test_model_inference(user_input:str):
    # 使用加载的模型 和 tokenizer进行基础推理测试
    messages = [
        {"role":"system","content":"You are Qwen, a helpful assistant."},
        {"role":"user","content":user_input}
    ]
    # 应用聊天模板，将对话消息转换为模型可以理解的格式
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,# 是否对输出的文本进行分词处理，如True：返回的是分词后的ID列表，False：返回的是格式化后的文本字符串
        add_generation_prompt=True# 指定是否在输出中添加生成提示
    )
    # 编码输入，并移动到设备
    inputs = tokenizer(text,return_tensors="pt").to(device)
    
    # 生成回答
    outs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample= True, # True：模型在生成每个token的时候会从概率分布中随机采样一个令牌，这样会导致生成的文本更加多样化和不可预测 False：模型直接选择概率最高的令牌，生成的文本更加确定和一致
        temperature=0.7
    )
    
    # 解码生成的的结果，取出特殊符号
    response = tokenizer.decode(outs[0],skip_special_tokens=True)
    return response

# 测试模型推理
test_intput = "how are you"
response = test_model_inference(test_intput)
print(f"测试输入：{test_intput}")
print(f"模型回答：{response}")

当前使用设备：cpu


## 将输入到模型中的数学题数据转换为模型能够理解的对话格式
- 该提示语的核心思想是：将推理过程和最终答案结构化分离，分别包裹在<think>和<answer>标签中，以便后续对不同部分分别进行评估和奖励。

In [1]:
# DeepSeek GRPO 系统提示
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, "
    "and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and "
    "then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> "
    "and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)

### 转换对话格式
- 原始数据中的每道题仅包含题干，我们需要把它包装成一段标准的对话


In [6]:
# 将原始样本转换为对话格式
def make_conversation(example):
    """我们将系统提示作为第一轮消息，再将题干作为用户提问，组成一段完整的对话轮次"""
    return {
        "prompt":[
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":example['problem']}
        ]
    }

### 加载并预处理数据集


In [8]:
from datasets import load_dataset

# 加载数学题数据集，并转换为对话格式
def load_math_dataset():
    dataset = load_dataset(
        "/Users/tao/gittao/datasets/NuminaMath-TIR",
        name="default",
        split=['train','test']
    )
    # 拆分成字典形式
    dataset = {
        'train':dataset[0],
        'test':dataset[1]
    }
    # 对训练集 和测试集进行格式转换
    for split  in dataset:
        dataset[split] = dataset[split].map(make_conversation)
        # 删除冗余字段
        if 'messages' in dataset[split].column_names:
            dataset[split] = dataset[split].remove_columns("messages")
    return dataset
dataset = load_math_dataset()
print(f"训练集大小:{len(dataset['train'])}")
print(f"测试集大小:{len(dataset['test'])}")

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

训练集大小:72441
测试集大小:99
